In [ ]:
# %%capture
# We're installing the latest Torch, Triton, OpenAI's Triton kernels, Transformers and Unsloth!
!pip install --upgrade -qqq uv
try: import numpy; install_numpy = f"numpy=={numpy.__version__}"
except: install_numpy = "numpy"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.5.0" {install_numpy} \
    "transformers>=4.56.0" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    torchvision bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 52.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load model using Unsloth FastLanguageModel for function calling
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Increased for function calling complexity
dtype = torch.float16
model_name = "meta-llama/Llama-3.2-3B-Instruct"  # Changed to Llama 3 for better function calling

# Load model and tokenizer with Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    dtype=dtype,
    max_seq_length=max_seq_length,
    load_in_4bit=True,  # Use 4-bit quantization to save memory
    full_finetuning=False,
)

print("✅ Model and tokenizer loaded successfully!")
print(f"Model: {model_name}")
print(f"Max sequence length: {max_seq_length}")
print(f"Using dtype: {dtype}")


In [ ]:
# ============================================================================
# UTILITY FUNCTIONS FOR FUNCTION CALLING FINE-TUNING WITH TOOL SUPPORT
# ============================================================================

import json
import re
import sys

def load_tools_from_definitions():
    """
    Load tool definitions from tool_definitions.py

    Returns:
        List of tool definitions or empty list if not found
    """
    try:
        # Try to import from the actual path
        sys.path.insert(0, '/content/drive/MyDrive/Colab Notebooks/AI/do an')
        from tool_definitions import INPUT_VALIDATION_TOOLS, INTENT_CLASSIFICATION_TOOLS, CLARIFICATION_TOOLS

        all_tools = []
        try:
            all_tools.extend(INPUT_VALIDATION_TOOLS if isinstance(INPUT_VALIDATION_TOOLS, list) else [INPUT_VALIDATION_TOOLS])
        except:
            pass
        try:
            all_tools.extend(INTENT_CLASSIFICATION_TOOLS if isinstance(INTENT_CLASSIFICATION_TOOLS, list) else [INTENT_CLASSIFICATION_TOOLS])
        except:
            pass
        try:
            all_tools.extend(CLARIFICATION_TOOLS if isinstance(CLARIFICATION_TOOLS, list) else [CLARIFICATION_TOOLS])
        except:
            pass

        print(f"✅ Loaded {len(all_tools)} tools from tool_definitions.py")
        return all_tools
    except Exception as e:
        print(f"⚠️ Could not load tools from tool_definitions.py: {e}")
        print("   Continuing without explicit tools...")
        return []


def extract_json_from_response(response: str) -> dict:
    """
    Extract JSON function call from model response

    Args:
        response: Model generated text

    Returns:
        Parsed JSON dict or None if no valid JSON found
    """
    # Try to find JSON block
    json_match = re.search(r'\{[^{}]*\}', response, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass
    return None


def validate_function_call(response: str) -> bool:
    """
    Check if response contains valid function call JSON

    Returns:
        True if valid JSON with 'name' and 'parameters' fields
    """
    try:
        data = extract_json_from_response(response)
        if data and "name" in data and "parameters" in data:
            return True
    except:
        pass
    return False


def handle_function_calling(query: str, tokenizer, model, tools=None):
    """
    Handle function calling for market research tasks with tool support

    Args:
        query: User input with tool definitions
        tokenizer: Tokenizer for model
        model: Fine-tuned model for function calling
        tools: Tool definitions (optional)

    Returns:
        dict: Parsed function call or error
    """
    try:
        messages = [
            {
                "role": "system",
                "content": "You are a market research analyst. You must respond with valid JSON function calls."
            },
            {
                "role": "user",
                "content": query
            }
        ]

        # Apply chat template with tools support (matching actual usage)
        if hasattr(tokenizer, "apply_chat_template"):
            model_input = tokenizer.apply_chat_template(
                messages,
                tools=tools,  # Pass tools if available
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            # Fallback: use last user message as prompt
            user_msgs = [m.get("content", "") for m in messages if m.get("role") == "user"]
            model_input = user_msgs[-1] if user_msgs else query

        # Tokenize
        model_inputs = tokenizer([model_input], return_tensors="pt")

        if torch.cuda.is_available():
            model_inputs = model_inputs.to('cuda')

        # Generate response
        with torch.no_grad():
            generated_ids = model.generate(
                model_inputs.input_ids,
                max_new_tokens=500,
                temperature=0.3,  # Low temperature for JSON precision
                top_p=0.95,
                do_sample=True,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
            )

        # Decode response
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # Extract and validate JSON
        json_data = extract_json_from_response(response)
        is_valid = json_data is not None and "name" in json_data and "parameters" in json_data

        return {
            "success": is_valid,
            "response": response,
            "json_data": json_data,
            "valid_function_call": is_valid
        }

    except Exception as e:
        return {
            "success": False,
            "error": str(e),
            "valid_function_call": False
        }


def test_function_calling(tokenizer, model, tools=None, test_cases=None):
    """
    Test function calling with sample test cases

    Args:
        tokenizer: Tokenizer
        model: Fine-tuned model
        tools: Tool definitions (optional)
        test_cases: List of test prompts (optional)
    """
    if test_cases is None:
        test_cases = [
            'Xin chào! Bạn khỏe không?',
            'Phân tích thị trường cà phê specialty ở Việt Nam',
        ]

    print("\n" + "="*70)
    print("🧪 TESTING FUNCTION CALLING CAPABILITY")
    print("="*70 + "\n")

    success_count = 0
    for i, test_prompt in enumerate(test_cases, 1):
        print(f"Test Case {i}: {test_prompt[:60]}...")
        print("-" * 70)

        result = handle_function_calling(test_prompt, tokenizer, model, tools=tools)

        if result["valid_function_call"]:
            success_count += 1
            print("✅ VALID FUNCTION CALL")
            print(f"Function: {result['json_data'].get('name')}")
            print(f"Parameters: {json.dumps(result['json_data'].get('parameters'), indent=2, ensure_ascii=False)}")
        else:
            print("❌ INVALID FUNCTION CALL")
            print(f"Response: {result['response'][:200]}...")

        print()

    print("="*70)
    print(f"✅ SUCCESS RATE: {success_count}/{len(test_cases)} ({100*success_count//len(test_cases)}%)")
    print("="*70 + "\n")

    return success_count / len(test_cases)


print("✅ Function calling utilities loaded successfully!")

✅ Function calling utilities loaded successfully!


In [ ]:
# ============================================================================
# SETUP LoRA FOR FUNCTION CALLING + LOAD TRAINING DATA WITH TOOL SUPPORT
# ============================================================================

# Setup LoRA optimized for function calling
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Higher rank for better function call accuracy
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,  # Higher alpha for stronger updates
    lora_dropout=0.0,  # Changed to 0 for Unsloth fast patching
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("✅ LoRA setup completed (optimized for function calling)!")

# Load ALL tool sets from tool_definitions (not just merged list)
print("\n🔧 Loading tool definitions...")
import sys
sys.path.insert(0, '/content/drive/MyDrive/Colab Notebooks/AI/do an')

try:
    from tool_definitions import (
        INPUT_VALIDATION_TOOLS,
        INTENT_CLASSIFICATION_TOOLS,
        CLARIFICATION_TOOLS,
        PLANNING_TOOLS,
        REACT_TOOLS,
        GENERATE_SEARCH_QUERIES_TOOLS,
        SYNTHESIS_TOOLS,
        CAMPAIGN_TOOLS,
        SWOT_TOOLS
    )

    # Create mapping of tool_calls names to actual tool definitions
    tool_sets = {
        "INPUT_VALIDATION_TOOLS": INPUT_VALIDATION_TOOLS,
        "INTENT_CLASSIFICATION_TOOLS": INTENT_CLASSIFICATION_TOOLS,
        "CLARIFICATION_TOOLS": CLARIFICATION_TOOLS,
        "PLANNING_TOOLS": PLANNING_TOOLS,
        "REACT_TOOLS": REACT_TOOLS,
        "GENERATE_SEARCH_QUERIES_TOOLS": GENERATE_SEARCH_QUERIES_TOOLS,
        "SYNTHESIS_TOOLS": SYNTHESIS_TOOLS,
        "CAMPAIGN_TOOLS": CAMPAIGN_TOOLS,
        "SWOT_TOOLS": SWOT_TOOLS,
    }
    print(f"✅ Loaded {len(tool_sets)} tool sets from tool_definitions.py")
    for name in tool_sets.keys():
        print(f"   - {name}")
except Exception as e:
    print(f"⚠️ Could not load tool sets: {e}")
    tool_sets = {}

# Load training data
import json
from datasets import load_dataset

print("\n📥 Loading training data...")

# Load from training_data.json (the standardized format)
with open("/content/drive/MyDrive/Colab Notebooks/AI/do an/training_data.json", "r", encoding="utf-8") as f:
    raw_training_data = json.load(f)

print(f"✅ Loaded {len(raw_training_data)} training examples")

# Convert to messages format with tool set mapping
training_examples = []
for example_idx, example in enumerate(raw_training_data):
    # Get messages
    messages = None
    if "messages" in example:
        messages = example.get("messages", [])
    elif "conversation" in example:
        messages = example.get("conversation", [])

    if messages and len(messages) > 0:
        # Extract tool_calls value from tool role message
        tools_for_this_example = None

        # Filter out role="tool" messages and extract tool set name
        filtered_messages = []
        for msg in messages:
            if msg.get("role") == "tool":
                # Extract tool_calls name for mapping
                tool_calls_name = msg.get("tool_calls", "")
                if tool_calls_name in tool_sets:
                    tools_for_this_example = tool_sets[tool_calls_name]
                # Don't add tool role message to filtered list!
            else:
                # Keep all non-tool messages (system, user, assistant)
                filtered_messages.append(msg)

        training_examples.append({
            "messages": filtered_messages,  # Messages WITHOUT tool role
            "tools": tools_for_this_example  # Tool set stored separately
        })

print(f"✅ Converted {len(training_examples)} examples with tool mappings")

if len(training_examples) == 0:
    print("❌ WARNING: No training examples found!")
else:
    # Create dataset
    from datasets import Dataset
    dataset = Dataset.from_dict({
        "text": ["" for _ in training_examples]  # Will be filled by formatting
    })

    print(f"✅ Dataset created successfully")
    print(f"Example 1 structure: {len(training_examples[0]['messages'])} messages (tool messages removed)")
    for i, msg in enumerate(training_examples[0]['messages']):
        role = msg.get('role', 'unknown')
        content_preview = msg.get('content', '')[:50] if msg.get('content') else 'N/A'
        print(f"  - Message {i}: {role} - {content_preview}...")

    # Format using apply_chat_template with correct tools for each example
    def format_function_calling(idx):
        """
        Format example using Llama chat template.
        Tool messages have been removed, so we can safely pass tools parameter.
        """
        messages = training_examples[idx]['messages']
        example_tools = training_examples[idx]['tools']

        # Apply chat template with the specific tools for this example
        if hasattr(tokenizer, "apply_chat_template"):
            try:
                text = tokenizer.apply_chat_template(
                    messages,
                    tools=example_tools if example_tools else None,  # Pass specific tools
                    tokenize=False,
                    add_generation_prompt=False
                )
            except Exception as e:
                print(f"   ⚠️ Error formatting example {idx}: {e}")
                # Fallback: format without tools (messages already cleaned)
                try:
                    text = tokenizer.apply_chat_template(
                        messages,
                        tokenize=False,
                        add_generation_prompt=False
                    )
                except Exception as e2:
                    print(f"   ❌ Fallback also failed: {e2}")
                    # Last resort: manual formatting
                    text = ""
                    for msg in messages:
                        role = msg.get('role', 'unknown')
                        content = msg.get('content', '')
                        text += f"<{role}>{content}</{role}>\n"
        else:
            # Fallback: basic formatting
            text = ""
            for msg in messages:
                role = msg.get('role', 'unknown')
                content = msg.get('content', '')
                text += f"<{role}>{content}</{role}>\n"

        return text

    print("\n🔄 Formatting dataset with per-example tools (tool messages removed)...")
    texts = [format_function_calling(i) for i in range(len(training_examples))]
    dataset = Dataset.from_dict({"text": texts})

    print("✅ Dataset formatted successfully!")
    print(f"Sample formatted text length: {len(dataset[0]['text'])} characters")
    print(f"First 300 chars:\n{dataset[0]['text'][:300]}...")

    # Setup training configuration
    from trl import SFTConfig, SFTTrainer

    training_config = SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=1e-4,
        logging_steps=1,
        max_steps=75,  # Stop training at maximum 75 steps
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="./llama_function_calling_finetune",
        report_to="none",
        save_strategy="no",
        eval_strategy="no",  # No evaluation - dataset too small
        max_grad_norm=1.0,
    )

    print("\n✅ Training configuration ready!")
    print(f"   - Batch size: {training_config.per_device_train_batch_size}")
    print(f"   - Learning rate: {training_config.learning_rate}")
    print(f"   - Epochs: {training_config.num_train_epochs}")
    print(f"   - Dataset size: {len(dataset)}")
    print(f"   - Tool sets available: {len(tool_sets)}")

    # Create trainer
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        args=training_config,
    )

    print("✅ SFTTrainer initialized successfully!")
    print("\n🚀 Ready to start training...")


In [ ]:
# ============================================================================
# START TRAINING WITH TOOL SUPPORT
# ============================================================================

print("\n" + "="*70)
print("🚀 STARTING FINE-TUNING WITH TOOL-AWARE FUNCTION CALLING")
print("="*70 + "\n")

# Start training
trainer.train()

print("\n" + "="*70)
print("✅ TRAINING COMPLETED!")
print("="*70)

# Save the model
output_dir = "./llama_function_calling_finetune_final"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"\n💾 Model saved to: {output_dir}")
print(f"   - Model weights saved")
print(f"   - Tokenizer saved")


In [ ]:
# ============================================================================
# EVALUATION AFTER TRAINING (POST-TRAINING)
# ============================================================================

print("\n" + "="*80)
print("🔍 POST-TRAINING EVALUATION")
print("="*80)
print("Evaluating fine-tuned model...\n")

# Put model back in eval mode
model.eval()

# Run evaluation on same validation set
post_training_metrics = evaluate_function_calling(
    model, 
    tokenizer, 
    validation_examples, 
    model_name="Fine-Tuned Model (After Training)",
    max_examples=len(validation_examples)
)

# Store for comparison
print("💾 Post-training metrics stored for comparison")


In [ ]:
# ============================================================================
# COMPARISON: BEFORE vs AFTER TRAINING
# ============================================================================

def compare_metrics(pre_metrics, post_metrics):
    """
    Compare evaluation metrics before and after training
    
    Args:
        pre_metrics: Metrics from pre-training evaluation
        post_metrics: Metrics from post-training evaluation
    """
    print("\n" + "="*80)
    print("📊 TRAINING IMPACT ANALYSIS: BEFORE vs AFTER")
    print("="*80)
    
    # Prepare comparison data
    comparisons = [
        ("Valid JSON Generation", pre_metrics["valid_json"], post_metrics["valid_json"], pre_metrics["total"]),
        ("'name' Field Presence", pre_metrics["has_name_field"], post_metrics["has_name_field"], pre_metrics["total"]),
        ("'parameters' Field Presence", pre_metrics["has_parameters_field"], post_metrics["has_parameters_field"], pre_metrics["total"]),
        ("'intent' Field Presence", pre_metrics["has_intent_field"], post_metrics["has_intent_field"], pre_metrics["total"]),
        ("Correct Intent Prediction", pre_metrics["correct_intent"], post_metrics["correct_intent"], pre_metrics["total"]),
        ("'reasoning' Field Presence", pre_metrics["has_reasoning_field"], post_metrics["has_reasoning_field"], pre_metrics["total"]),
    ]
    
    print(f"\n{'Metric':<35} {'Before':<15} {'After':<15} {'Improvement':<15}")
    print("─" * 80)
    
    total_improvement = 0
    
    for metric_name, pre_value, post_value, total in comparisons:
        pre_pct = (pre_value / total * 100) if total > 0 else 0
        post_pct = (post_value / total * 100) if total > 0 else 0
        improvement = post_pct - pre_pct
        total_improvement += improvement
        
        improvement_str = f"+{improvement:.1f}%" if improvement >= 0 else f"{improvement:.1f}%"
        print(f"{metric_name:<35} {pre_pct:>6.1f}%        {post_pct:>6.1f}%        {improvement_str:>10}")
    
    print("─" * 80)
    print(f"\n{'Overall Improvements':<35}")
    print(f"   • Average improvement across metrics: +{total_improvement/len(comparisons):.2f}%")
    print(f"   • Pre-training overall accuracy: {pre_metrics['overall_accuracy']:.1f}%")
    print(f"   • Post-training overall accuracy: {post_metrics['overall_accuracy']:.1f}%")
    print(f"   • Accuracy gain: +{post_metrics['overall_accuracy'] - pre_metrics['overall_accuracy']:.1f}%")
    
    print(f"\n{'Intent Classification Accuracy':<35}")
    print(f"   • Before: {pre_metrics['intent_accuracy']:.1f}%")
    print(f"   • After: {post_metrics['intent_accuracy']:.1f}%")
    print(f"   • Improvement: +{post_metrics['intent_accuracy'] - pre_metrics['intent_accuracy']:.1f}%")
    
    # Errors
    print(f"\n{'Error Handling':<35}")
    print(f"   • Errors before training: {pre_metrics['error_count']}")
    print(f"   • Errors after training: {post_metrics['error_count']}")
    print(f"   • Error reduction: {pre_metrics['error_count'] - post_metrics['error_count']}")
    
    print(f"\n{'='*80}\n")
    
    # Create detailed comparison table
    print("📋 DETAILED INTENT-BY-INTENT COMPARISON:\n")
    
    all_intents = set(pre_metrics["accuracy_by_intent"].keys()) | set(post_metrics["accuracy_by_intent"].keys())
    
    if all_intents:
        print(f"{'Intent':<20} {'Before':<15} {'After':<15} {'Improvement':<15}")
        print("─" * 65)
        
        for intent in sorted(all_intents):
            pre_stats = pre_metrics["accuracy_by_intent"].get(intent, {"total": 0, "correct": 0})
            post_stats = post_metrics["accuracy_by_intent"].get(intent, {"total": 0, "correct": 0})
            
            pre_acc = (pre_stats["correct"] / pre_stats["total"] * 100) if pre_stats["total"] > 0 else 0
            post_acc = (post_stats["correct"] / post_stats["total"] * 100) if post_stats["total"] > 0 else 0
            improvement = post_acc - pre_acc
            
            improvement_str = f"+{improvement:.1f}%" if improvement >= 0 else f"{improvement:.1f}%"
            
            # Show count for reference
            pre_str = f"{pre_acc:.1f}% ({pre_stats['correct']}/{pre_stats['total']})"
            post_str = f"{post_acc:.1f}% ({post_stats['correct']}/{post_stats['total']})"
            
            print(f"{intent:<20} {pre_str:<15} {post_str:<15} {improvement_str:>10}")
    
    print(f"\n{'='*80}\n")


# Run comparison if both metrics exist
if 'pre_training_metrics' in locals() and 'post_training_metrics' in locals():
    compare_metrics(pre_training_metrics, post_training_metrics)
    
    # Save comparison to file
    comparison_results = {
        "pre_training": {k: v for k, v in pre_training_metrics.items() if k != "errors"},
        "post_training": {k: v for k, v in post_training_metrics.items() if k != "errors"},
        "improvement": {
            "overall_accuracy_gain": post_training_metrics["overall_accuracy"] - pre_training_metrics["overall_accuracy"],
            "intent_accuracy_gain": post_training_metrics["intent_accuracy"] - pre_training_metrics["intent_accuracy"],
        }
    }
    
    comparison_file = "/content/drive/MyDrive/Colab Notebooks/AI/do an/evaluation_comparison.json"
    with open(comparison_file, "w", encoding="utf-8") as f:
        json.dump(comparison_results, f, indent=2, ensure_ascii=False)
    
    print(f"💾 Comparison results saved to: {comparison_file}")
else:
    print("⚠️ Both pre-training and post-training metrics needed for comparison")
    print("   Please run pre-training evaluation first, then training, then post-training evaluation")


In [ ]:
# ============================================================================
# LOAD VALIDATION DATA AND PREPARE FOR EVALUATION
# ============================================================================

import json
import re
from collections import defaultdict

print("📥 Loading validation data...")

# Load validate_data.json
with open("/content/drive/MyDrive/Colab Notebooks/AI/do an/validate_data.json", "r", encoding="utf-8") as f:
    validation_data = json.load(f)

print(f"✅ Loaded {len(validation_data)} validation examples")

# Parse validation examples
validation_examples = []
for example_idx, example in enumerate(validation_data):
    messages = example.get("messages", [])
    
    if len(messages) >= 4:  # Ensure we have system, user, tool, assistant
        system_msg = None
        user_msg = None
        assistant_msg = None
        expected_intent = None
        expected_response = None
        
        for msg in messages:
            role = msg.get("role")
            if role == "system":
                system_msg = msg.get("content", "")
            elif role == "user":
                user_msg = msg.get("content", "")
            elif role == "assistant":
                content = msg.get("content", "")
                assistant_msg = content
                
                # Extract expected intent and response from assistant message
                try:
                    json_data = json.loads(content)
                    expected_intent = json_data.get("arguments", {}).get("intent")
                    expected_response = json_data.get("arguments", {}).get("response", "")
                except:
                    pass
        
        if user_msg and assistant_msg:
            validation_examples.append({
                "user_input": user_msg,
                "full_messages": messages,
                "assistant_response": assistant_msg,
                "expected_intent": expected_intent,
                "expected_response": expected_response,
                "system_prompt": system_msg
            })

print(f"✅ Parsed {len(validation_examples)} validation examples\n")

# Show sample examples
print("Sample validation examples:")
for i in range(min(3, len(validation_examples))):
    print(f"\nExample {i+1}:")
    print(f"  User: {validation_examples[i]['user_input'][:60]}...")
    print(f"  Expected intent: {validation_examples[i]['expected_intent']}")
    print(f"  Assistant response preview: {validation_examples[i]['assistant_response'][:80]}...")

# Create intent distribution
intent_distribution = defaultdict(int)
for ex in validation_examples:
    intent = ex.get("expected_intent")
    if intent:
        intent_distribution[intent] += 1

print(f"\n📊 Intent distribution in validation set:")
for intent, count in sorted(intent_distribution.items(), key=lambda x: x[1], reverse=True):
    print(f"   - {intent}: {count} examples ({count*100//len(validation_examples)}%)")


In [ ]:
# ============================================================================
# EVALUATION FUNCTION FOR FUNCTION CALLING
# ============================================================================

def evaluate_function_calling(model, tokenizer, validation_examples, model_name="Model", max_examples=None):
    """
    Evaluate function calling accuracy on validation set
    
    Args:
        model: The model to evaluate
        tokenizer: Model tokenizer
        validation_examples: List of validation examples
        model_name: Name for display
        max_examples: Maximum examples to evaluate (None = all)
    
    Returns:
        dict: Evaluation metrics
    """
    import torch
    
    if max_examples is None:
        max_examples = len(validation_examples)
    else:
        max_examples = min(max_examples, len(validation_examples))
    
    metrics = {
        "total": max_examples,
        "valid_json": 0,
        "has_name_field": 0,
        "has_parameters_field": 0,
        "has_intent_field": 0,
        "correct_intent": 0,
        "has_reasoning_field": 0,
        "accuracy_by_intent": defaultdict(lambda: {"total": 0, "correct": 0}),
        "error_count": 0,
        "errors": []
    }
    
    print(f"\n{'='*80}")
    print(f"🧪 EVALUATING {model_name}")
    print(f"{'='*80}")
    print(f"Evaluating on {max_examples} examples...\n")
    
    for idx in range(max_examples):
        example = validation_examples[idx]
        user_input = example["user_input"]
        system_prompt = example.get("system_prompt", "")
        expected_intent = example.get("expected_intent")
        
        try:
            # Prepare messages for model
            messages = [
                {
                    "role": "system",
                    "content": system_prompt if system_prompt else "You are a market research analyst. Classify user intent and respond appropriately."
                },
                {
                    "role": "user",
                    "content": user_input
                }
            ]
            
            # Apply chat template
            if hasattr(tokenizer, "apply_chat_template"):
                model_input = tokenizer.apply_chat_template(
                    messages,
                    tools=None,
                    tokenize=False,
                    add_generation_prompt=True,
                )
            else:
                model_input = user_input
            
            # Tokenize
            model_inputs = tokenizer([model_input], return_tensors="pt")
            
            if torch.cuda.is_available():
                model_inputs = model_inputs.to('cuda')
            
            # Generate response
            with torch.no_grad():
                generated_ids = model.generate(
                    model_inputs.input_ids,
                    max_new_tokens=300,
                    temperature=0.3,
                    top_p=0.95,
                    do_sample=False,  # Use deterministic for evaluation
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.eos_token_id,
                )
            
            # Decode response
            generated_ids = [
                output_ids[len(input_ids):]
                for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
            ]
            
            response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            
            # Extract JSON from response
            json_match = re.search(r'\{[^{}]*\}', response, re.DOTALL)
            json_data = None
            
            if json_match:
                try:
                    json_data = json.loads(json_match.group())
                    metrics["valid_json"] += 1
                except:
                    pass
            
            # Check field presence
            if json_data:
                if "name" in json_data:
                    metrics["has_name_field"] += 1
                if "parameters" in json_data:
                    metrics["has_parameters_field"] += 1
                    params = json_data["parameters"]
                    if "intent" in params:
                        metrics["has_intent_field"] += 1
                        predicted_intent = params["intent"]
                        
                        # Track by intent
                        if expected_intent:
                            metrics["accuracy_by_intent"][expected_intent]["total"] += 1
                            if predicted_intent == expected_intent:
                                metrics["correct_intent"] += 1
                                metrics["accuracy_by_intent"][expected_intent]["correct"] += 1
                    
                    if "reasoning" in params:
                        metrics["has_reasoning_field"] += 1
        
        except Exception as e:
            metrics["error_count"] += 1
            metrics["errors"].append({
                "example_idx": idx,
                "user_input": user_input[:50],
                "error": str(e)
            })
        
        # Progress indicator
        if (idx + 1) % max(1, max_examples // 10) == 0:
            print(f"  Processed {idx + 1}/{max_examples} examples...")
    
    # Calculate accuracy
    accuracy = metrics["valid_json"] / metrics["total"] * 100 if metrics["total"] > 0 else 0
    intent_accuracy = metrics["correct_intent"] / metrics["total"] * 100 if metrics["total"] > 0 else 0
    
    print(f"\n{'='*80}")
    print(f"📊 EVALUATION RESULTS FOR {model_name}")
    print(f"{'='*80}")
    print(f"\n📈 Overall Metrics:")
    print(f"   • Total examples: {metrics['total']}")
    print(f"   • Valid JSON: {metrics['valid_json']}/{metrics['total']} ({accuracy:.1f}%)")
    print(f"   • Has 'name' field: {metrics['has_name_field']}/{metrics['total']}")
    print(f"   • Has 'parameters' field: {metrics['has_parameters_field']}/{metrics['total']}")
    print(f"   • Has 'intent' field: {metrics['has_intent_field']}/{metrics['total']}")
    print(f"   • Correct intent prediction: {metrics['correct_intent']}/{metrics['total']} ({intent_accuracy:.1f}%)")
    print(f"   • Has 'reasoning' field: {metrics['has_reasoning_field']}/{metrics['total']}")
    print(f"   • Errors: {metrics['error_count']}")
    
    if metrics["accuracy_by_intent"]:
        print(f"\n🎯 Accuracy by Intent:")
        for intent, stats in sorted(metrics["accuracy_by_intent"].items()):
            acc = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
            print(f"   • {intent}: {stats['correct']}/{stats['total']} ({acc:.1f}%)")
    
    print(f"\n{'='*80}\n")
    
    metrics["overall_accuracy"] = accuracy
    metrics["intent_accuracy"] = intent_accuracy
    
    return metrics


print("✅ Evaluation function defined successfully!")


In [ ]:
# ============================================================================
# EVALUATION BEFORE TRAINING (PRE-TRAINING BASELINE)
# ============================================================================

print("\n" + "="*80)
print("🔍 PRE-TRAINING EVALUATION")
print("="*80)
print("Evaluating base model before fine-tuning...\n")

# Prepare model for inference
model.eval()

# Run evaluation on validation set (using first 20 examples for speed)
pre_training_metrics = evaluate_function_calling(
    model, 
    tokenizer, 
    validation_examples, 
    model_name="Base Model (Before Training)",
    max_examples=len(validation_examples)  # Evaluate all examples
)

# Store for later comparison
print("💾 Pre-training metrics stored for comparison")


In [ ]:
# ============================================================================
# PUSH FINE-TUNED MODEL TO HUGGING FACE HUB
# ============================================================================

from huggingface_hub import login, HfApi

print("\n" + "="*70)
print("🚀 PUSHING MODEL TO HUGGING FACE HUB")
print("="*70 + "\n")

# Login to Hugging Face (you need a write token)
# Get token from: https://huggingface.co/settings/tokens
print("📝 Logging into Hugging Face...")
print("   (You need a write access token from https://huggingface.co/settings/tokens)")

try:
    login("")
    print("✅ Logged in successfully!")
except Exception as e:
    print(f"⚠️ Login failed: {e}")
    print("   Skipping Hugging Face push...")
    exit()

# Push model to hub
repo_id = "justrandomname/function-calling"
print(f"\n📤 Pushing model to {repo_id}...")

try:
    # Push model weights
    model.push_to_hub(
        repo_id=repo_id,
        token=None,  # Uses login() token
        private=False,
        commit_message="Fine-tuned Llama-3.2-3B for function calling with tool support"
    )
    print(f"✅ Model pushed successfully!")

    # Push tokenizer
    print(f"📤 Pushing tokenizer...")
    tokenizer.push_to_hub(
        repo_id=repo_id,
        token=None,
        private=False,
    )
    print(f"✅ Tokenizer pushed successfully!")

    # Push training config
    print(f"📤 Pushing training config...")
    training_config.push_to_hub(
        repo_id=repo_id,
        token=None,
    )
    print(f"✅ Training config pushed successfully!")

    print("\n" + "="*70)
    print(f"🎉 SUCCESS! Model available at:")
    print(f"   https://huggingface.co/{repo_id}")
    print("="*70)

except Exception as e:
    print(f"❌ Push failed: {e}")
    print("   Make sure your token has write access to the repo")


## 📊 Function Calling Evaluation Guide

### Evaluation Workflow

The notebook now includes comprehensive evaluation cells to measure function calling performance before and after training:

1. **Load Validation Data** - Loads `validate_data.json` with 32 test examples
2. **Define Evaluation Function** - Creates `evaluate_function_calling()` for detailed metrics
3. **Pre-Training Evaluation** - Baseline performance on base model
4. **Training** - Fine-tune on training data
5. **Post-Training Evaluation** - Performance after fine-tuning
6. **Comparison Analysis** - Side-by-side metrics and improvements

### Metrics Tracked

- **Valid JSON**: Percentage of responses with valid JSON structure
- **Field Presence**: Detection of required fields (name, parameters, intent, reasoning)
- **Intent Accuracy**: Correct classification of user intent
- **Accuracy by Intent**: Per-intent breakdown (research, chat, knowledge)

### Output Files

- **evaluation_comparison.json**: Saved to Google Drive with pre/post metrics and improvements

### Expected Improvements

Typical improvements after fine-tuning:
- Valid JSON accuracy: baseline → 85-95%
- Intent accuracy: baseline → 70-85%
- Field presence: significant improvement in structured output


In [ ]:
!cp -r "./llama_function_calling_finetune_final" "/content/drive/MyDrive/Colab Notebooks/AI/do an/"